In [1]:
from datasets import load_dataset
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

C:\Users\ADMIN\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load vocab

In [2]:
vocab = []
with open(r"D:\NLP\project\vocabulary.txt", 'r', encoding='utf-8') as f:
    vocab = f.read().splitlines()

# Lọc các từ giống nhau
vocab = list(dict.fromkeys(vocab))

# Ánh xạ word và idx
word_to_idx = {word: i for i, word in enumerate(vocab)}

Load stopwords

In [3]:
with open(r"D:\NLP\project\vietnamese-stopwords.txt", 'r', encoding='utf-8') as f:
    stopwords = f.read().splitlines()
stopword = set(stopwords)

Load dataset

In [4]:
dataset = load_dataset("yammdd/vietnamese-error-correction-corpus")
df_train = pd.DataFrame(dataset['train'])

Tạo các cặp skipgram với window_size = 5

In [5]:
window_size = 5
skipgram_data = []

for sentence in df_train['target']:
    # Lấy danh sách từ
    sentence = sentence.lower()
    words = sentence.split()

    # Lọc stopword
    filtered_words = []
    for word in words:
        if word not in word_to_idx or word in stopword:
            continue

        filtered_words.append(word)

    for i, word in enumerate(filtered_words):
        # Kiểm tra xem từ này có trong vocab
        if word not in word_to_idx:
            continue

        center = word_to_idx[word]

        for j in range(-window_size, window_size + 1):
            # Kiểm tra các khoảng cách xem có phù hợp (trừ chính nó)
            if j == 0:
                continue
            if 0 <= i + j < len(filtered_words):
                context_word = words[i + j]
                #Kiểm tra các từ bên cạnh có trong vocab không
                if context_word in word_to_idx:
                    context = word_to_idx[context_word]
                    skipgram_data.append((center, context))

In [6]:
len(skipgram_data)

1996340

In [7]:
class SkipGram(nn.Module):

    def __init__(self, vocab_size, embed_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):

        embed = self.embedding(x)
        out = self.linear(embed)

        return out

In [13]:
#  Chuẩn bị chạy bằng GPU
device = torch.device("cuda")

centers = torch.LongTensor([pair[0] for pair in skipgram_data])
contexts = torch.LongTensor([pair[1] for pair in skipgram_data])

batch_size = 8192
dataset = TensorDataset(centers, contexts)
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

EMBED_DIM = 300
VOCAB_SIZE = len(vocab)
model_skipgram = SkipGram(VOCAB_SIZE, EMBED_DIM).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_skipgram.parameters(), lr=0.001)

In [ ]:
epochs = 30

model_skipgram.train()

for epoch in range(epochs):
    total_loss = 0

    for batch_centers, batch_contexts in loader:

        batch_centers = batch_centers.to(device)
        batch_contexts = batch_contexts.to(device)

        # Forward
        outputs = model_skipgram(batch_centers)
        loss = criterion(outputs, batch_contexts)

        # Backward
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 2070.0903
Epoch 2, Loss: 1634.1326
Epoch 3, Loss: 1604.1827
Epoch 4, Loss: 1589.4813
Epoch 5, Loss: 1579.7079
Epoch 6, Loss: 1572.5143
Epoch 7, Loss: 1566.7907
Epoch 8, Loss: 1562.0750
Epoch 9, Loss: 1557.9741
Epoch 10, Loss: 1554.4234
Epoch 11, Loss: 1551.2863
Epoch 12, Loss: 1548.4097
Epoch 13, Loss: 1545.7901
Epoch 14, Loss: 1543.4017
Epoch 15, Loss: 1541.1702
Epoch 16, Loss: 1539.1074
Epoch 17, Loss: 1537.1604
Epoch 18, Loss: 1535.3403
Epoch 19, Loss: 1533.5965
Epoch 20, Loss: 1531.9565
Epoch 21, Loss: 1530.4060
Epoch 22, Loss: 1528.9458
Epoch 23, Loss: 1527.5280
Epoch 24, Loss: 1526.2009
Epoch 25, Loss: 1524.8747
Epoch 26, Loss: 1523.6405
Epoch 27, Loss: 1522.4366
Epoch 28, Loss: 1521.3057
Epoch 29, Loss: 1520.1528
Epoch 30, Loss: 1519.1027


Lưu model train skip-gram

In [ ]:
torch.save(model_skipgram.state_dict(), 'model_skipgram.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>